# 🔧 Predictive Maintenance — Dataset Exploration

Notebook ini melakukan **Exploratory Data Analysis (EDA)** lengkap pada dataset  
**Predictive Maintenance Classification** milik PT Volex.

## Tujuan Notebook
1. **Memahami struktur** dataset predictive maintenance
2. **Analisis variabel target** — `failure_next_24h` & `maintenance_risk_level`
3. **Eksplorasi fitur** — distribusi, korelasi, pola temporal
4. **Analisis class imbalance** — distribusi kelas target
5. **Insight** — fitur paling berpengaruh untuk model klasifikasi

---
| Properti | Detail |
|---|---|
| **File** | `predictive maintenance dataset.xlsx` |
| **Rows** | 6,340 records |
| **Kolom** | 29 kolom |
| **Target 1** | `failure_next_24h` → Binary Classification (Yes/No) |
| **Target 2** | `maintenance_risk_level` → Multi-class (No/Low/Medium/High Risk) |
| **Periode** | Feb 2026 – Aug 2026 |

---
## 📦 1. Import Library & Load Dataset

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

plt.rcParams.update({
    'figure.figsize'   : (14, 5),
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'font.size'        : 11,
})
sns.set_palette('tab10')

# ── Path ──────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATASET_PATH = PROJECT_ROOT / 'dataset' / 'predictive maintenance dataset.xlsx'

assert DATASET_PATH.exists(), f'❌ File tidak ditemukan: {DATASET_PATH}'
print('✅ Import sukses!')
print(f'📂 Dataset: {DATASET_PATH}')

In [ ]:
# Load dataset
df = pd.read_excel(DATASET_PATH)

print(f'✅ Dataset berhasil diload!')
print(f'   Shape  : {df.shape[0]:,} rows × {df.shape[1]} kolom')
print(f'   Memori : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print()
df.head()

---
## 🔍 2. Profiling Kolom

In [ ]:
# Profiling lengkap semua kolom
profile = pd.DataFrame({
    'dtype'    : df.dtypes,
    'non_null' : df.notna().sum(),
    'null'     : df.isna().sum(),
    'null_%'   : (df.isna().mean() * 100).round(2),
    'unique'   : df.nunique(),
    'sample'   : df.apply(lambda c: str(c.dropna().iloc[0])[:50] if c.notna().any() else 'ALL NULL'),
})

print('📋 Profil Semua Kolom:')
print(profile.to_string())

In [ ]:
# Kelompokkan kolom berdasarkan fungsinya
COL_GROUPS = {
    'Identifier'     : ['record_id', 'timestamp', 'record_date', 'machine_id',
                         'machine_name', 'line_id', 'line_name', 'line_process_id', 'shift_id'],
    'Produksi'       : ['total_output', 'avg_cycle_time_sec', 'throughput_units_per_hour',
                         'shift_total_pass', 'operating_seconds'],
    'OEE'            : ['oee', 'availability', 'performance', 'quality'],
    'Kualitas'       : ['total_produced', 'total_pass', 'total_reject', 'defect_rate'],
    'Mesin/Maint.'   : ['cum_running_hours', 'days_since_last_maintenance',
                         'corrective_wo_last_7d', 'preventive_wo_last_30d'],
    'Target'         : ['hours_to_next_failure_event', 'failure_next_24h', 'maintenance_risk_level'],
}

print('📋 Kelompok Kolom:')
for group, cols in COL_GROUPS.items():
    print(f'\n  [{group}]  ({len(cols)} kolom)')
    for col in cols:
        if col in df.columns:
            print(f'    · {col} ({df[col].dtype})')

---
## 🎯 3. Analisis Variabel Target

In [ ]:
# ── 3.1 Target 1: failure_next_24h (Binary Classification) ───────────────────
target_bin = df['failure_next_24h'].value_counts()
target_bin_pct = df['failure_next_24h'].value_counts(normalize=True) * 100

print('=== TARGET 1: failure_next_24h ===')
print(f'  Tipe : Binary Classification (Yes / No)')
print(f'  Total: {len(df):,} records')
for val in target_bin.index:
    print(f'  {val:<5}: {target_bin[val]:>5,} ({target_bin_pct[val]:.1f}%)')
print(f'  Class Imbalance Ratio: 1 : {target_bin["No"] / target_bin["Yes"]:.1f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors_bin = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(target_bin.index, target_bin.values, color=colors_bin, alpha=0.85, edgecolor='white')
axes[0].bar_label(bars, labels=[f'{v:,}\n({p:.1f}%)' for v, p in zip(target_bin.values, target_bin_pct.values)],
                   padding=5, fontsize=11)
axes[0].set_title('Distribusi failure_next_24h\n(Target Binary Classification)')
axes[0].set_xlabel('Kelas')
axes[0].set_ylabel('Jumlah Record')
axes[0].set_ylim(0, target_bin.max() * 1.2)

# Pie chart
axes[1].pie(target_bin.values, labels=target_bin.index,
            autopct='%1.1f%%', colors=colors_bin,
            startangle=90, explode=[0, 0.05])
axes[1].set_title('Proporsi Kelas failure_next_24h')

plt.suptitle('⚠️ Class Imbalance: "No" jauh lebih banyak dari "Yes"',
             fontsize=12, color='darkorange', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.2 Target 2: maintenance_risk_level (Multi-class Classification) ─────────
RISK_ORDER  = ['No Risk', 'Low Risk', 'Medium Risk', 'High Risk']
RISK_COLORS = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']

target_mc     = df['maintenance_risk_level'].value_counts().reindex(RISK_ORDER)
target_mc_pct = df['maintenance_risk_level'].value_counts(normalize=True).reindex(RISK_ORDER) * 100

print('=== TARGET 2: maintenance_risk_level ===')
print(f'  Tipe : Multi-class Classification (4 kelas)')
for val in RISK_ORDER:
    n   = target_mc[val] if val in target_mc.index else 0
    pct = target_mc_pct[val] if val in target_mc_pct.index else 0
    print(f'  {val:<15}: {n:>5,} ({pct:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(RISK_ORDER, target_mc.values, color=RISK_COLORS, alpha=0.85, edgecolor='white')
axes[0].bar_label(bars, labels=[f'{v:,}\n({p:.1f}%)' for v, p in zip(target_mc.values, target_mc_pct.values)],
                   padding=5, fontsize=10)
axes[0].set_title('Distribusi maintenance_risk_level\n(Target Multi-class Classification)')
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Jumlah Record')
axes[0].set_ylim(0, target_mc.max() * 1.2)

# Stacked horizontal
total = len(df)
left  = 0
for val, color in zip(RISK_ORDER, RISK_COLORS):
    w = target_mc[val] / total * 100
    axes[1].barh(['Risk Level'], [w], left=left, color=color, label=val, height=0.5)
    if w > 2:
        axes[1].text(left + w / 2, 0, f'{w:.1f}%', ha='center', va='center',
                     fontsize=10, color='white', fontweight='bold')
    left += w
axes[1].set_xlim(0, 100)
axes[1].set_xlabel('Proporsi (%)')
axes[1].set_title('Komposisi Kelas Risk Level')
axes[1].legend(loc='upper right', bbox_to_anchor=(1, 1.3))
axes[1].set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3 Cek Konsistensi antara Dua Target ────────────────────────────────────
cross = pd.crosstab(df['failure_next_24h'], df['maintenance_risk_level'],
                     margins=True, margins_name='Total')
print('📊 Cross-tabulation: failure_next_24h vs maintenance_risk_level')
print(cross)

# Visualisasi
fig, ax = plt.subplots(figsize=(10, 5))
cross_pct = pd.crosstab(df['failure_next_24h'], df['maintenance_risk_level'], normalize='index') * 100
cross_pct = cross_pct.reindex(columns=RISK_ORDER, fill_value=0)
cross_pct.plot(kind='bar', ax=ax, color=RISK_COLORS, alpha=0.85, edgecolor='white')
ax.set_title('Proporsi Risk Level per Kelas failure_next_24h')
ax.set_xlabel('failure_next_24h')
ax.set_ylabel('Proporsi (%)')
ax.legend(title='Risk Level', bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('\n💡 Insight:')
print('   → "Yes" failure seharusnya berkorelasi dengan High/Medium Risk')
print('   → "No"  failure seharusnya berkorelasi dengan No/Low Risk')

In [ ]:
# ── 3.4 Target Regresi: hours_to_next_failure_event ──────────────────────────
htnf = df['hours_to_next_failure_event'].dropna()
print('=== TARGET REGRESI: hours_to_next_failure_event ===')
print(f'  Jumlah valid : {len(htnf):,} ({len(htnf)/len(df)*100:.1f}%)')
print(f'  Missing      : {df["hours_to_next_failure_event"].isna().sum()} (last window, normal)')
print(f'  Min  : {htnf.min():.2f} jam')
print(f'  Max  : {htnf.max():.2f} jam')
print(f'  Mean : {htnf.mean():.2f} jam')
print(f'  Median: {htnf.median():.2f} jam')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(htnf, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(24,            color='red',    linestyle='--', linewidth=2, label='24 jam (threshold)')
axes[0].axvline(htnf.mean(),   color='orange', linestyle='--', linewidth=2, label=f'Mean: {htnf.mean():.1f} jam')
axes[0].axvline(htnf.median(), color='green',  linestyle=':',  linewidth=2, label=f'Median: {htnf.median():.1f} jam')
axes[0].set_title('Distribusi hours_to_next_failure_event')
axes[0].set_xlabel('Jam ke Failure Berikutnya')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

# Distribusi per risk level
for risk, color in zip(RISK_ORDER, RISK_COLORS):
    data = df[df['maintenance_risk_level'] == risk]['hours_to_next_failure_event'].dropna()
    if len(data):
        axes[1].hist(data, bins=30, alpha=0.6, label=f'{risk} (n={len(data)})', color=color)

axes[1].axvline(24, color='black', linestyle='--', linewidth=2, label='24 jam threshold')
axes[1].set_title('Distribusi hours_to_next_failure per Risk Level')
axes[1].set_xlabel('Jam ke Failure')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 🏭 4. Analisis Fitur — Mesin & Operasional

In [ ]:
# ── 4.1 Distribusi per Mesin ──────────────────────────────────────────────────
machine_stats = df.groupby('machine_name').agg(
    n_records         = ('record_id',             'count'),
    failure_rate      = ('failure_next_24h',       lambda x: (x == 'Yes').mean() * 100),
    avg_oee           = ('oee',                   'mean'),
    avg_cum_hours     = ('cum_running_hours',      'mean'),
    avg_days_since_maint = ('days_since_last_maintenance', 'mean'),
    avg_corrective_wo = ('corrective_wo_last_7d',  'mean'),
).round(3).sort_values('failure_rate', ascending=False)

print('📊 Statistik per Mesin (sorted by failure_rate):')
print(machine_stats.to_string())

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

machine_sorted = machine_stats.sort_values('failure_rate', ascending=True)
colors = sns.color_palette('RdYlGn_r', len(machine_sorted))

bars = axes[0].barh(machine_sorted.index, machine_sorted['failure_rate'], color=colors)
axes[0].bar_label(bars, fmt='{:.1f}%', padding=3)
axes[0].set_title('Failure Rate (%) per Mesin')
axes[0].set_xlabel('Failure Rate (%)')

machine_oee = machine_stats.sort_values('avg_oee', ascending=True)
bars2 = axes[1].barh(machine_oee.index, machine_oee['avg_oee'], color='steelblue')
axes[1].bar_label(bars2, fmt='{:.3f}', padding=3)
axes[1].set_title('Rata-rata OEE per Mesin')
axes[1].set_xlabel('Avg OEE')
axes[1].axvline(0.85, color='red', linestyle='--', linewidth=1.5, label='World-class (0.85)')
axes[1].legend(fontsize=9)

machine_wo = machine_stats.sort_values('avg_corrective_wo', ascending=True)
bars3 = axes[2].barh(machine_wo.index, machine_wo['avg_corrective_wo'], color='salmon')
axes[2].bar_label(bars3, fmt='{:.2f}', padding=3)
axes[2].set_title('Avg Corrective WO per 7 Hari')
axes[2].set_xlabel('Avg Corrective WO')

plt.suptitle('Perbandingan Metrik per Mesin', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.2 Cumulative Running Hours vs Failure ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box plot: cum_running_hours per failure class
fail_yes = df[df['failure_next_24h'] == 'Yes']['cum_running_hours']
fail_no  = df[df['failure_next_24h'] == 'No']['cum_running_hours']
axes[0].boxplot([fail_no, fail_yes], tick_labels=['No Failure', 'Will Fail'],
                 patch_artist=True,
                 boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[0].set_title('Cumulative Running Hours\nvs Failure Next 24h')
axes[0].set_ylabel('Cum. Running Hours')
t_stat, p_val = stats.ttest_ind(fail_yes, fail_no)
axes[0].text(0.5, 0.95, f'p-value: {p_val:.4f} ({"signifikan" if p_val < 0.05 else "tidak signifikan"})',
             transform=axes[0].transAxes, ha='center', va='top',
             color='red' if p_val < 0.05 else 'gray', fontsize=10)

# Scatter: cum_hours vs days_since_maintenance, warna per risk level
for risk, color in zip(RISK_ORDER, RISK_COLORS):
    subset = df[df['maintenance_risk_level'] == risk]
    axes[1].scatter(subset['cum_running_hours'], subset['days_since_last_maintenance'],
                    alpha=0.3, s=15, color=color, label=risk)
axes[1].set_title('Cum. Running Hours vs Days Since Last Maintenance\n(warna = Risk Level)')
axes[1].set_xlabel('Cumulative Running Hours')
axes[1].set_ylabel('Days Since Last Maintenance')
axes[1].legend(title='Risk Level', markerscale=2)

plt.tight_layout()
plt.show()

print('\n💡 Statistik Cum. Running Hours per Kelas:')
print(df.groupby('failure_next_24h')['cum_running_hours'].describe().round(2).to_string())

In [ ]:
# ── 4.3 Work Order Analysis ───────────────────────────────────────────────────
print('📊 Work Order Statistics:')
print('\nCorrective WO Last 7 Days:')
print(df['corrective_wo_last_7d'].value_counts().sort_index().to_string())
print('\nPreventive WO Last 30 Days:')
print(df['preventive_wo_last_30d'].value_counts().sort_index().to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# WO vs failure rate
corr_wo_fail = df.groupby('corrective_wo_last_7d')['failure_next_24h'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index()
corr_wo_fail.columns = ['corrective_wo', 'failure_rate']
axes[0].bar(corr_wo_fail['corrective_wo'], corr_wo_fail['failure_rate'],
             color=sns.color_palette('YlOrRd', len(corr_wo_fail)), edgecolor='white')
axes[0].set_title('Failure Rate (%) vs Corrective WO Last 7 Days')
axes[0].set_xlabel('Jumlah Corrective WO (7 hari terakhir)')
axes[0].set_ylabel('Failure Rate (%)')

# Preventive WO vs risk level
prev_risk = df.groupby(['preventive_wo_last_30d', 'maintenance_risk_level']).size().unstack(fill_value=0)
prev_risk = prev_risk.reindex(columns=RISK_ORDER, fill_value=0)
prev_risk.plot(kind='bar', ax=axes[1], color=RISK_COLORS, alpha=0.85, edgecolor='white')
axes[1].set_title('Risk Level vs Preventive WO Last 30 Days')
axes[1].set_xlabel('Jumlah Preventive WO (30 hari terakhir)')
axes[1].set_ylabel('Jumlah Record')
axes[1].legend(title='Risk Level')
plt.setp(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

---
## 📈 5. Analisis OEE & Performa terhadap Failure

In [ ]:
# ── 5.1 OEE Components vs Failure ────────────────────────────────────────────
oee_cols = ['oee', 'availability', 'performance', 'quality']

print('📊 OEE Stats per Failure Class:')
print(df.groupby('failure_next_24h')[oee_cols].mean().round(4).to_string())

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, col in zip(axes, oee_cols):
    data_groups = [df[df['failure_next_24h'] == cls][col].dropna() for cls in ['No', 'Yes']]
    bp = ax.boxplot(data_groups, tick_labels=['No', 'Yes'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor('#e74c3c')
    bp['boxes'][1].set_alpha(0.7)

    t_stat, p_val = stats.ttest_ind(data_groups[1], data_groups[0])
    sig_mark = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
    ax.set_title(f'{col}\np={p_val:.3f} {sig_mark}')
    ax.set_xlabel('failure_next_24h')
    ax.set_ylabel(col)

plt.suptitle('OEE Components vs failure_next_24h\n(* = signifikan, *** = sangat signifikan)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2 OEE per Risk Level ────────────────────────────────────────────────────
print('📊 OEE Stats per Risk Level:')
print(df.groupby('maintenance_risk_level')[oee_cols].mean().round(4)
       .reindex(RISK_ORDER).to_string())

fig, ax = plt.subplots(figsize=(12, 6))
oee_risk = df.groupby('maintenance_risk_level')[oee_cols].mean().reindex(RISK_ORDER)
x = np.arange(len(RISK_ORDER))
width = 0.2
bar_colors = ['#3498db', '#27ae60', '#e67e22', '#9b59b6']

for i, (col, color) in enumerate(zip(oee_cols, bar_colors)):
    bars = ax.bar(x + i * width, oee_risk[col], width, label=col.upper(),
                  color=color, alpha=0.8, edgecolor='white')

ax.set_title('Rata-rata OEE Components per Risk Level')
ax.set_xlabel('Risk Level')
ax.set_ylabel('Nilai')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(RISK_ORDER)
ax.legend(title='Metrik')
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

---
## ⏰ 6. Analisis Temporal

In [ ]:
# ── 6.1 Tren Failure Rate dari Waktu ke Waktu ─────────────────────────────────
df['record_date_dt'] = pd.to_datetime(df['record_date'])
df['year_month']     = df['record_date_dt'].dt.to_period('W').astype(str)

weekly_fail = df.groupby('year_month').agg(
    n_records     = ('record_id', 'count'),
    n_failure     = ('failure_next_24h', lambda x: (x == 'Yes').sum()),
    failure_rate  = ('failure_next_24h', lambda x: (x == 'Yes').mean() * 100),
    avg_oee       = ('oee', 'mean'),
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Failure rate mingguan
axes[0].plot(range(len(weekly_fail)), weekly_fail['failure_rate'],
              color='#e74c3c', linewidth=2, marker='o', markersize=4)
axes[0].fill_between(range(len(weekly_fail)), weekly_fail['failure_rate'],
                      alpha=0.15, color='#e74c3c')
axes[0].axhline(weekly_fail['failure_rate'].mean(), color='orange', linestyle='--',
                 linewidth=1.5, label=f'Rata-rata: {weekly_fail["failure_rate"].mean():.1f}%')
axes[0].set_title('Tren Failure Rate Mingguan (%)')
axes[0].set_ylabel('Failure Rate (%)')
axes[0].set_xticks(range(0, len(weekly_fail), 4))
axes[0].set_xticklabels(weekly_fail['year_month'].iloc[::4], rotation=45, ha='right', fontsize=8)
axes[0].legend()

# OEE mingguan
axes[1].plot(range(len(weekly_fail)), weekly_fail['avg_oee'],
              color='steelblue', linewidth=2, marker='o', markersize=4)
axes[1].fill_between(range(len(weekly_fail)), weekly_fail['avg_oee'],
                      alpha=0.15, color='steelblue')
axes[1].axhline(0.85, color='red', linestyle='--', linewidth=1.5, label='World-class OEE (85%)')
axes[1].set_title('Tren OEE Mingguan')
axes[1].set_ylabel('OEE')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
axes[1].set_xticks(range(0, len(weekly_fail), 4))
axes[1].set_xticklabels(weekly_fail['year_month'].iloc[::4], rotation=45, ha='right', fontsize=8)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 6.2 Failure Rate per Shift ────────────────────────────────────────────────
SHIFT_MAP = {
    '4a28e93f-183c-4b5f-9e90-703d80626680': 'Shift 1 (06-14)',
    '15b09269-ea40-4cc7-aeca-d0c85b53e3da': 'Shift 2 (14-22)',
    'b1b71e03-f8a2-456d-b6a6-389839d0bdd4': 'Shift 3 (22-06)',
}
df['shift_name'] = df['shift_id'].map(SHIFT_MAP).fillna(df['shift_id'])

shift_fail = df.groupby('shift_name').agg(
    n_records    = ('record_id', 'count'),
    failure_rate = ('failure_next_24h', lambda x: (x == 'Yes').mean() * 100),
    avg_oee      = ('oee', 'mean'),
).round(2)

print('📊 Failure Rate per Shift:')
print(shift_fail.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
shifts = shift_fail.index.tolist()
bars = axes[0].bar(shifts, shift_fail['failure_rate'],
                    color=['#3498db', '#e74c3c', '#2ecc71'], alpha=0.85)
axes[0].bar_label(bars, fmt='{:.1f}%', padding=5)
axes[0].set_title('Failure Rate (%) per Shift')
axes[0].set_ylabel('Failure Rate (%)')
plt.setp(axes[0].get_xticklabels(), rotation=15)

bars2 = axes[1].bar(shifts, shift_fail['avg_oee'],
                     color=['#3498db', '#e74c3c', '#2ecc71'], alpha=0.85)
axes[1].bar_label(bars2, fmt='{:.3f}', padding=5)
axes[1].axhline(0.85, color='red', linestyle='--', linewidth=1.5, label='Target 85%')
axes[1].set_title('Rata-rata OEE per Shift')
axes[1].set_ylabel('Avg OEE')
axes[1].legend()
plt.setp(axes[1].get_xticklabels(), rotation=15)

plt.tight_layout()
plt.show()

---
## 🔗 7. Analisis Korelasi & Feature Importance

In [ ]:
# ── 7.1 Korelasi antar Fitur Numerik ──────────────────────────────────────────
NUM_FEATURE_COLS = [
    'total_output', 'avg_cycle_time_sec', 'throughput_units_per_hour',
    'shift_total_pass', 'operating_seconds',
    'oee', 'availability', 'performance', 'quality',
    'total_produced', 'total_pass', 'total_reject', 'defect_rate',
    'cum_running_hours', 'days_since_last_maintenance',
    'corrective_wo_last_7d', 'preventive_wo_last_30d',
    'hours_to_next_failure_event',
]
NUM_FEATURE_COLS = [c for c in NUM_FEATURE_COLS if c in df.columns]

corr_matrix = df[NUM_FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
             center=0, vmin=-1, vmax=1, ax=ax,
             linewidths=0.3, annot_kws={'size': 8},
             square=True)
ax.set_title('Korelasi Antar Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.2 Korelasi Fitur vs Target (Point-Biserial untuk Binary) ─────────────────
from scipy.stats import pointbiserialr

df['failure_binary'] = (df['failure_next_24h'] == 'Yes').astype(int)

corr_with_target = {}
for col in NUM_FEATURE_COLS:
    valid_mask = df[col].notna() & df['failure_binary'].notna()
    if valid_mask.sum() > 10:
        corr_val, p_val = pointbiserialr(df.loc[valid_mask, 'failure_binary'],
                                          df.loc[valid_mask, col])
        corr_with_target[col] = {'corr': round(corr_val, 4), 'p_value': round(p_val, 6)}

target_corr_df = pd.DataFrame(corr_with_target).T.sort_values('corr', key=abs, ascending=False)

print('📊 Korelasi Fitur dengan failure_next_24h (Point-Biserial):')
print(target_corr_df.to_string())

# Visualisasi
fig, ax = plt.subplots(figsize=(12, 8))
colors_bar = ['#e74c3c' if v > 0 else '#2ecc71' for v in target_corr_df['corr']]
bars = ax.barh(target_corr_df.index, target_corr_df['corr'], color=colors_bar, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Korelasi Fitur vs failure_next_24h\n(Merah = korelasi positif dengan failure, Hijau = negatif)')
ax.set_xlabel('Point-Biserial Correlation')

# Tandai fitur yang signifikan
for i, (col, row) in enumerate(target_corr_df.iterrows()):
    sig = '***' if row['p_value'] < 0.001 else ('*' if row['p_value'] < 0.05 else '')
    if sig:
        ax.text(row['corr'] + (0.003 if row['corr'] >= 0 else -0.003),
                i, sig, va='center', ha='left' if row['corr'] >= 0 else 'right',
                fontsize=10, color='navy')

plt.tight_layout()
plt.show()

---
## ⚖️ 8. Analisis Missing Values & Class Imbalance

In [ ]:
# ── 8.1 Missing Values Summary ────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = df.isnull().mean() * 100
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct.round(2)})
missing_df = missing_df[missing_df['count'] > 0].sort_values('count', ascending=False)

print('📋 Kolom dengan Missing Values:')
if len(missing_df):
    print(missing_df.to_string())
    print()
    for col in missing_df.index:
        print(f'  · {col}: {int(missing_df.loc[col,"count"])} null '
              f'({missing_df.loc[col,"pct"]:.1f}%) → '
              + (
                  'Wajar: mesin belum pernah maintenance' if 'maintenance' in col
                  else ('Wajar: tidak selalu ada event failure' if 'failure' in col
                  else 'Perlu investigasi')
              ))
else:
    print('   ✅ Tidak ada missing values!')

In [ ]:
# ── 8.2 Analisis Class Imbalance & Rekomendasi ───────────────────────────────
print('=' * 60)
print('  ANALISIS CLASS IMBALANCE')
print('=' * 60)

# Binary
n_no  = (df['failure_next_24h'] == 'No').sum()
n_yes = (df['failure_next_24h'] == 'Yes').sum()
ratio_bin = n_no / n_yes
print(f'\n[failure_next_24h]')
print(f'  No  : {n_no:,} ({n_no/len(df)*100:.1f}%)')
print(f'  Yes : {n_yes:,} ({n_yes/len(df)*100:.1f}%)')
print(f'  Ratio: 1 : {ratio_bin:.1f} (imbalanced!)')
print(f'  → Rekomendasi: SMOTE / Class Weight / Threshold Tuning')

# Multi-class
print(f'\n[maintenance_risk_level]')
for risk in RISK_ORDER:
    n = (df['maintenance_risk_level'] == risk).sum()
    print(f'  {risk:<15}: {n:,} ({n/len(df)*100:.1f}%)')
print(f'  → Rekomendasi: Balanced class weights / SMOTE-NC')

# Visualisasi imbalance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Binary
wedge_props = {'edgecolor': 'white', 'linewidth': 2}
axes[0].pie([n_no, n_yes], labels=['No (80.3%)', 'Yes (19.7%)'],
             colors=['#2ecc71', '#e74c3c'],
             autopct='%1.1f%%', startangle=90, wedgeprops=wedge_props,
             explode=[0, 0.1])
axes[0].set_title('Class Distribution\nfailure_next_24h')

# Multi-class
mc_counts = [df[df['maintenance_risk_level'] == r].shape[0] for r in RISK_ORDER]
axes[1].pie(mc_counts, labels=[f'{r}\n({v:,})' for r, v in zip(RISK_ORDER, mc_counts)],
             colors=RISK_COLORS, autopct='%1.1f%%', startangle=90,
             wedgeprops=wedge_props)
axes[1].set_title('Class Distribution\nmaintenance_risk_level')

plt.suptitle('⚠️ Class Imbalance — Perlu Penanganan Sebelum Training',
             color='darkorange', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔑 9. Top Feature Analysis per Risk Level

In [ ]:
# ── 9.1 Distribusi Fitur Kunci per Risk Level ─────────────────────────────────
KEY_FEATURES = [
    'cum_running_hours', 'days_since_last_maintenance',
    'corrective_wo_last_7d', 'oee', 'availability',
    'defect_rate', 'hours_to_next_failure_event',
]
KEY_FEATURES = [f for f in KEY_FEATURES if f in df.columns]

n = len(KEY_FEATURES)
cpr = 4
fig, axes = plt.subplots((n + cpr - 1) // cpr, cpr, figsize=(20, 10))
axes = axes.flat

for ax, feat in zip(axes, KEY_FEATURES):
    for risk, color in zip(RISK_ORDER, RISK_COLORS):
        data = df[df['maintenance_risk_level'] == risk][feat].dropna()
        if len(data):
            ax.hist(data, bins=25, alpha=0.5, color=color, label=risk, density=True)
    ax.set_title(feat)
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)

for ax in list(axes)[len(KEY_FEATURES):]:
    ax.set_visible(False)

plt.suptitle('Distribusi Fitur Kunci per Risk Level (Density Plot)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.2 Tabel Statistik Fitur per Risk Level ──────────────────────────────────
print('📊 Mean Fitur Kunci per Risk Level:')
feat_by_risk = df.groupby('maintenance_risk_level')[KEY_FEATURES].mean().reindex(RISK_ORDER).round(3)
print(feat_by_risk.to_string())

# Heatmap normalized
feat_norm = (feat_by_risk - feat_by_risk.min()) / (feat_by_risk.max() - feat_by_risk.min())

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(feat_norm, annot=feat_by_risk.values, fmt='.2f',
             cmap='YlOrRd', ax=ax, linewidths=0.5,
             annot_kws={'size': 9}, cbar_kws={'label': 'Normalized Value'})
ax.set_title('Heatmap Fitur Kunci per Risk Level\n(nilai aktual, warna dinormalisasi)',
              fontsize=12, fontweight='bold')
ax.set_xlabel('Fitur')
ax.set_ylabel('Risk Level')
plt.tight_layout()
plt.show()

---
## 💾 10. Simpan Dataset & Ringkasan

In [ ]:
# Simpan dataset ke folder processed
OUTPUT_DIR = PROJECT_ROOT / 'dataset' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset lengkap (raw)
path_raw = OUTPUT_DIR / 'predictive_maintenance_raw.csv'
df.drop(columns=['failure_binary', 'shift_name', 'year_month', 'record_date_dt'],
         errors='ignore').to_csv(path_raw, index=False)
print(f'✅ Dataset raw tersimpan : {path_raw.name}  ({df.shape})')

# Dataset untuk Binary Classification
FEATURE_COLS_MODEL = [
    'total_output', 'avg_cycle_time_sec', 'throughput_units_per_hour',
    'shift_total_pass', 'operating_seconds',
    'oee', 'availability', 'performance', 'quality',
    'total_produced', 'total_pass', 'total_reject', 'defect_rate',
    'cum_running_hours', 'days_since_last_maintenance',
    'corrective_wo_last_7d', 'preventive_wo_last_30d',
]
FEATURE_COLS_MODEL = [c for c in FEATURE_COLS_MODEL if c in df.columns]

# Binary Classification
df_binary = df[FEATURE_COLS_MODEL + ['failure_next_24h']].dropna()
path_bin  = OUTPUT_DIR / 'pm_binary_classification.csv'
df_binary.to_csv(path_bin, index=False)
print(f'✅ Binary Classification  : {path_bin.name}  ({df_binary.shape})')

# Multi-class Classification
df_multi = df[FEATURE_COLS_MODEL + ['maintenance_risk_level']].dropna()
path_mc  = OUTPUT_DIR / 'pm_multiclass_classification.csv'
df_multi.to_csv(path_mc, index=False)
print(f'✅ Multi-class Classification: {path_mc.name}  ({df_multi.shape})')

# Regression (hours_to_next_failure)
df_reg  = df[FEATURE_COLS_MODEL + ['hours_to_next_failure_event']].dropna()
path_reg = OUTPUT_DIR / 'pm_regression_hours_to_failure.csv'
df_reg.to_csv(path_reg, index=False)
print(f'✅ Regression              : {path_reg.name}  ({df_reg.shape})')

print(f'\n📂 Semua file tersimpan di: {OUTPUT_DIR}')

In [ ]:
print('=' * 70)
print('  RINGKASAN EDA — PREDICTIVE MAINTENANCE DATASET')
print('=' * 70)
print(f'''
📌 OVERVIEW
  • Records        : {len(df):,}
  • Fitur numerik  : {len(FEATURE_COLS_MODEL)}
  • Periode        : {df["record_date"].min().date()} s/d {df["record_date"].max().date()}
  • Mesin          : {df["machine_name"].nunique()} mesin
  • Line           : {df["line_name"].nunique()} line

📌 TARGET VARIABEL
  1. failure_next_24h     → Binary Classification
     • No  : {n_no:,} ({n_no/len(df)*100:.1f}%)
     • Yes : {n_yes:,} ({n_yes/len(df)*100:.1f}%)
     • ⚠️  Imbalanced 1:{ratio_bin:.0f} → Gunakan SMOTE / class_weight

  2. maintenance_risk_level → Multi-class Classification
     • No Risk    : {(df["maintenance_risk_level"]=="No Risk").sum():,}
     • Low Risk   : {(df["maintenance_risk_level"]=="Low Risk").sum():,}
     • Medium Risk: {(df["maintenance_risk_level"]=="Medium Risk").sum():,}
     • High Risk  : {(df["maintenance_risk_level"]=="High Risk").sum():,}
     • ⚠️  Imbalanced → Gunakan balanced class weights

  3. hours_to_next_failure_event → Regression
     • Mean : {df["hours_to_next_failure_event"].mean():.1f} jam
     • Range: {df["hours_to_next_failure_event"].min():.0f} – {df["hours_to_next_failure_event"].max():.0f} jam
     • Missing: {df["hours_to_next_failure_event"].isna().sum()} rows (normal)

📌 FITUR PALING RELEVAN (untuk failure prediction)
  1. corrective_wo_last_7d      → Makin banyak WO → makin berisiko
  2. cum_running_hours          → Jam jalan kumulatif mesin
  3. days_since_last_maintenance→ Makin lama tidak maintenance → berisiko
  4. oee / performance          → OEE rendah → indikasi masalah
  5. defect_rate                → Defect tinggi → mesin bermasalah
  6. availability               → Availability rendah → downtime sering
  7. hours_to_next_failure_event→ Ground truth waktu ke failure

📌 CATATAN PENTING
  • Missing values wajar: days_since_last_maintenance (264 null = mesin baru)
  • Missing values wajar: hours_to_next_failure_event (123 null = window akhir)
  • Perlu SMOTE atau class_weight untuk mengatasi class imbalance
  • Dataset sudah clean — siap untuk data cleaning notebook

📌 OUTPUT FILES
  • predictive_maintenance_raw.csv      → Full dataset
  • pm_binary_classification.csv        → Features + failure_next_24h
  • pm_multiclass_classification.csv    → Features + maintenance_risk_level
  • pm_regression_hours_to_failure.csv  → Features + hours_to_next_failure
''')
print('=' * 70)
print('  ✅ EDA SELESAI — DATASET SIAP UNTUK DATA CLEANING & MODEL TRAINING!')
print('=' * 70)